# Cleanup — Tear Down Demo Resources

Removes all resources created by the demo that `databricks bundle destroy` cannot handle:
- Synced tables
- Lakebase project
- Model Serving endpoint
- UC registered model
- Schemas and tables

**Run `databricks bundle destroy` AFTER this notebook to remove jobs, pipeline, and app.**

In [ ]:
%pip install -U "databricks-sdk>=0.81.0"
dbutils.library.restartPython()

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

CATALOG = "startups_catalog"
PROJECT_ID = "db-residential-copilot"
LAKEBASE_CATALOG = f"{PROJECT_ID}.production.databricks_postgres"
ENDPOINT_NAME = "investment_copilot"
MODEL_NAME = f"{CATALOG}.gold.investment_copilot"

## 1. Delete Synced Tables

In [ ]:
synced_tables = [
    f"{LAKEBASE_CATALOG}.app.portfolio_metrics",
    f"{LAKEBASE_CATALOG}.app.portfolio_time_series",
]

for table_name in synced_tables:
    try:
        w.database.delete_synced_database_table(name=table_name)
        print(f"Deleted synced table: {table_name}")
    except Exception as e:
        print(f"Skip {table_name}: {e}")

## 2. Delete Model Serving Endpoint

In [ ]:
try:
    w.serving_endpoints.delete(ENDPOINT_NAME)
    print(f"Deleted serving endpoint: {ENDPOINT_NAME}")
except Exception as e:
    print(f"Skip endpoint: {e}")

## 3. Delete UC Registered Model

In [ ]:
try:
    from mlflow import MlflowClient
    client = MlflowClient(registry_uri="databricks-uc")
    client.delete_registered_model(MODEL_NAME)
    print(f"Deleted registered model: {MODEL_NAME}")
except Exception as e:
    print(f"Skip model: {e}")

## 4. Delete Lakebase Project

In [ ]:
try:
    w.postgres.delete_project(name=f"projects/{PROJECT_ID}")
    print(f"Deleted Lakebase project: {PROJECT_ID}")
except Exception as e:
    print(f"Skip Lakebase project: {e}")

## 5. Drop Schemas (gold, silver, bronze, raw)

**Caution:** This drops all tables in these schemas. Only run if you want a full teardown.

In [ ]:
schemas_to_drop = ["gold", "silver", "bronze", "raw"]

for schema in schemas_to_drop:
    try:
        spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{schema} CASCADE")
        print(f"Dropped schema: {CATALOG}.{schema}")
    except Exception as e:
        print(f"Skip schema {schema}: {e}")

print("\nCleanup complete. Now run: databricks bundle destroy -t dev -p vm")